In [ ]:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)


In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)



print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))


In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
X_batch_sq = X_batch.unsqueeze(-1)
print(f"Training batch input shape: {X_batch_sq.shape}")


In [ ]:
# 5. Display sample images
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze())
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):
    """
    A three-layer fully connected neural network for image classification.
    Architecture: Input(784) -> Hidden(hidden_dim) -> Hidden(hidden_dim) -> Output(10)
    """

    def __init__(self, input_dim, hidden_dim, output_dim):
        """
        Initialize the network layers.

        Parameters:
            input_dim: Number of input features (784 for flattened 28x28 images)
            hidden_dim: Number of neurons in hidden layers
            output_dim: Number of output classes (10 for digits 0-9)
        """
        # Call parent class constructor to initialize nn.Module internals
        super(NN3Layer, self).__init__()

        # First fully connected layer: transforms input features to hidden representation
        # nn.Linear(in_features, out_features) performs: output = input @ weight.T + bias
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second fully connected layer: hidden layer to hidden layer
        # Deeper networks can learn more complex patterns
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer to number of classes (logits)
        # Outputs raw scores, NOT probabilities (softmax applied in loss function)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        self.layer4 = nn.Linear(hidden_dim, output_dim)


        # ReLU activation function for non-linearity between layers
        # ReLU(x) = max(0, x) - simple and effective for deep networks
        self.relu = nn.ReLU()

    def forward(self, x):
        """
        Define the forward pass - how data flows through the network.

        Parameters:
            x: Input tensor of shape (batch_size, 784)
        Returns:
            Output tensor of shape (batch_size, 10) - raw logits
        """
        # First hidden layer: linear transformation + activation
        z1 = self.layer1(x)    # Linear: (batch, 784) -> (batch, hidden_dim)
        a1 = self.relu(z1)     # Activation: apply non-linearity

        # Second hidden layer: linear transformation + activation
        z2 = self.layer2(a1)   # Linear: (batch, hidden_dim) -> (batch, hidden_dim)
        a2 = self.relu(z2)     # Activation: apply non-linearity

        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        # Output layer: raw scores (logits) for each class
        # NOTE: No softmax here - CrossEntropyLoss applies it internally
        output = self.layer4(a3)  # Linear: (batch, hidden_dim) -> (batch, 10)

        return output
        #Copy paste just added a layer !!

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)         # shape: (batch_size, num_features)
        y_batch = y_batch.to(device)         # shape: (batch_size,)

        # Forward pass (outputs are logits)
        outputs = model(X_batch)             # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)     # shape: (batch_size, num_features)
            y_batch = y_batch.to(device)     # shape: (batch_size,)

            # Forward pass
            outputs = model(X_batch)         # shape: (batch_size, num_classes)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # Pick the classes with highest probabilities
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:

In [ ]:
# Task 5: Start training for 20 epochs:

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: